# **Setup and process train model machine learning**

**1. Boosted Hybrid Model**

Create a boosted hybrid model for training.

In [ ]:
import lightgbm as lgb
model_hybrid = BoostedHybrid( model_1 = LinearRegression() , model_2 = lgb.LGBMRegressor(n_estimators = 3000 , learning_rate = 0.05 , max_depth = 10 , n_jobs = -1 , num_leaves = 127))
model_hybrid.fit(X_train_1, X_train_2 , y_train_hd )

Then, we will perform forecasting on the validation set.

In [ ]:
y_pred_valid_hd = model_hybrid.predict(X_valid_1 , X_valid_2)
y_pred_valid_hd = y_pred_valid_hd.clip(0.0)
y_pred_test_hd = model_hybrid.predict(X_test_1 , X_test_2)
y_pred_test_hd = y_pred_test_hd.clip(0.0)
y_valid_hd = np.expm1(y_valid_hd)
y_pred_valid_hd = np.expm1(y_pred_valid_hd)
y_test_hd = np.expm1(y_test_hd)
y_pred_test_hd = np.expm1(y_pred_test_hd)

print('complete!')

representing the learning and prediction processes of the hybrid model.

In [ ]:
y_sample = y_train_hd['Sales'][:100].values
y_fit_sample = model_hybrid.y_fit['Sales'][:100].values
y_res = model_hybrid.y_resid.unstack()
y_resid_sample = y_res['Sales'][:100].values
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
axes[0].plot( y_sample , label='Actual Revenue', color='gray', alpha=0.6)
axes[0].plot(y_fit_sample, label='The Projected Revenue Of Model_1', color='red', linewidth=2)
axes[0].set_title('Linear Regressor Line')
axes[0].legend()

axes[1].plot( y_resid_sample ,  label='Unexplained Residual', color='orange')
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_title('Residual For LGBM Regressor')
axes[1].legend()

y_valid_sample = y_valid_hd[:100].values
y_pred_valid_sample = y_pred_valid_hd[:100].values

axes[2].plot(y_valid_sample, label='Actual(Valid)', color='gray', alpha=0.7)
axes[2].plot(y_pred_valid_sample, label='Forecast from the complete hybrid model', color='green', linewidth=2)
axes[2].set_title('Consolidated revenue forecast based on two models')
axes[2].legend()
plt.tight_layout()

**2. Single LGBM Model**

set up the machine learning model and adjust the necessary parameters of model

In [ ]:
import lightgbm as lgb
model_lgb = lgb.LGBMRegressor( n_estimators = 3000 , learning_rate = 0.05 , max_depth = 10 , n_jobs = -1 , num_leaves = 127 )
model_lgb.fit( X_train, y_train, 
           eval_set=[(X_valid, y_valid)],
           callbacks=[lgb.early_stopping(stopping_rounds=50) , lgb.log_evaluation(period=200)]
)

visualize the model's learning process

In [ ]:
lgb.plot_metric(model_lgb, metric='l2', figsize=(10, 6))
plt.title('Chart of model learning process (Learning Curve)')
plt.xlabel('Number of Trees (Iterations)')
plt.ylabel('Error MSE (l2)')
plt.show()

machine learning model testing

In [ ]:
from sklearn.metrics import r2_score , mean_absolute_error , mean_squared_error
my_pre_val = model_lgb.predict(X_valid)
my_pre_test = model_lgb.predict(X_test)
y_valid = np.expm1(y_valid)
y_test = np.expm1(y_test)
my_pre_val = np.expm1(my_pre_val)
my_pre_test = np.expm1(my_pre_test)
print('Complete!')

the process of learning and prediction on the datasets.

In [ ]:
fig, axes = plt.subplots( nrows = 3 , ncols = 1 , figsize = (15 , 12 ) , sharex = True )
y_train_sample = y_train[:100].values
y_train_pred = model_lgb.predict(X_train)
y_train_pred_sample = y_train_pred[:100]
y_valid_sample = y_valid[:100].values
y_valid_pred_sample = my_pre_val[:100]
y_test_sample = y_test[:100].values
y_test_pred_sample = my_pre_test[:100]
axes[0].plot( y_train_sample , label = 'Actual(Train set)' , color = 'gray' , alpha = 0.7)
axes[0].plot( y_train_pred_sample , label = 'Forecasting of LGBM model on the train set' , color = 'red' , linewidth = 2 )
axes[0].set_title('The training and prediction process of the LGBM model on the training set')
axes[0].legend()

axes[1].plot( y_valid_sample , label = 'Actual(Valid set)' , color = 'gray' , alpha = 0.7)
axes[1].plot( y_valid_pred_sample , label = 'Forecasting of LGBM model on the valid set' , color = 'blue' , linewidth = 2 )
axes[1].set_title('The training and prediction process of the LGBM model on the validition set')
axes[1].legend()

axes[2].plot( y_test_sample , label = 'Actual(Test set)' , color = 'gray' , alpha = 0.7)
axes[2].plot( y_test_pred_sample , label = 'Forecasting of LGBM model on the test set' , color = 'green' , linewidth = 2 )
axes[2].set_title('The training and prediction process of the LGBM model on the test set')
axes[2].legend()
plt.tight_layout()